CN7023-SEMCD-14_UdayKiranMudu

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower, regexp_replace
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import sys # Import sys to exit if file is not found

# ==========================================
# STEP 1: Spark Setup & Dataset Loading
# ==========================================
spark = SparkSession.builder \
    .appName("Enron_Email_Spam_Classification_NaiveBayes") \
    .master("local[*]") \
    .getOrCreate()

# Load Enron Email Dataset
dataset_path = "emails.csv"  # Ensure emails.csv is in your working directory

# --- BEGIN FIX ---
# Check if the dataset exists. If not, inform the user and exit.
if not os.path.exists(dataset_path):
    print(f"ERROR: The dataset '{dataset_path}' was not found in the current directory (/content/).")
    print("Please upload 'emails.csv' to your Colab environment. You can do this by:")
    print("1. Clicking the folder icon on the left-hand side panel.")
    print("2. Clicking the 'Upload to session storage' icon (a document with an arrow pointing up).")
    print("3. Selecting 'emails.csv' from your local machine.")
    print("Alternatively, if 'emails.csv' is located elsewhere, please update the 'dataset_path' variable with the correct path.")
    sys.exit("Dataset not found. Please upload the file or correct the path.")
# --- END FIX ---

# Reading CSV file
df = spark.read.csv(dataset_path, header=True, inferSchema=True)

# Standardize schema (rename columns if necessary)
# Assuming dataset has 'text' and 'spam' (or 'label') columns
if "spam" in df.columns:
    df = df.withColumnRenamed("spam", "label_raw")
elif "label" in df.columns:
    df = df.withColumnRenamed("label", "label_raw")

# Select and clean base columns
df = df.select(col("text"), col("label_raw").cast("integer").alias("label_raw"))

# Basic text cleaning: lowercase and remove non-alphanumeric characters
df_cleaned = df.withColumn("text_clean", lower(col("text"))) \
               .withColumn("text_clean", regexp_replace(col("text_clean"), "[^a-zA-Z0-9\\s]", "")) \
               .dropna(subset=["text_clean", "label_raw"])

In [ ]:
# ==========================================
# STEP 2: Text Preprocessing & Feature Extraction (TF-IDF)
# ==========================================
# Pipeline Stages:
# 1. Label Indexer (ensures target is binary numerical 0.0 or 1.0)
# 2. Tokenizer (splits text into words)
# 3. StopWordsRemover (removes noise words like 'the', 'is')
# 4. HashingTF (term frequency counts)
# 5. IDF (inverse document frequency weighting)

indexer = StringIndexer(inputCol="label_raw", outputCol="label")
tokenizer = Tokenizer(inputCol="text_clean", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=10000)
idf = IDF(inputCol="raw_features", outputCol="features")

# Fit & Transform Preprocessing Pipeline
prep_pipeline = Pipeline(stages=[indexer, tokenizer, remover, hashingTF, idf])
prep_model = prep_pipeline.fit(df_cleaned)
processed_df = prep_model.transform(df_cleaned)

In [ ]:
# ==========================================
# STEP 3: Train / Test Split
# ==========================================
train_data, test_data = processed_df.randomSplit([0.8, 0.2], seed=42)
print(f"Training Count: {train_data.count()}, Testing Count: {test_data.count()}")

Training Count: 4194, Testing Count: 978


In [ ]:
# ==========================================
# STEP 4: Model Training & Comparison of 3 Naïve Bayes Variants
# ==========================================
nb_variants = {
    "Multinomial Naïve Bayes": NaiveBayes(modelType="multinomial", featuresCol="features", labelCol="label"),
    "Bernoulli Naïve Bayes": NaiveBayes(modelType="bernoulli", featuresCol="features", labelCol="label"),
    "Complement Naïve Bayes": NaiveBayes(modelType="complement", featuresCol="features", labelCol="label")
}

def evaluate_model(predictions):
    eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    eval_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    eval_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
    eval_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

    return {
        "Accuracy": eval_acc.evaluate(predictions),
        "Precision": eval_prec.evaluate(predictions),
        "Recall": eval_rec.evaluate(predictions),
        "F1-Score": eval_f1.evaluate(predictions)
    }

results = {}

# Import Binarizer for Bernoulli Naïve Bayes
from pyspark.ml.feature import Binarizer

for model_name, nb_instance in nb_variants.items():
    print(f"\nTraining {model_name}...")

    current_train_data = train_data
    current_test_data = test_data

    # Bernoulli Naïve Bayes requires binary features (0 or 1).
    # The 'raw_features' column contains term frequencies which can be binarized.
    if model_name == "Bernoulli Naïve Bayes":
        binarizer = Binarizer(inputCol="raw_features", outputCol="binary_features", threshold=0.0)
        current_train_data = binarizer.transform(train_data)
        current_test_data = binarizer.transform(test_data)
        # Update the NaiveBayes instance's featuresCol for this iteration
        nb_instance.setFeaturesCol("binary_features")

    model = nb_instance.fit(current_train_data)
    predictions = model.transform(current_test_data)
    results[model_name] = evaluate_model(predictions)

    # Reset featuresCol for the original NaiveBayes instance for subsequent iterations if needed
    # (though in this loop, nb_instance is reassigned each time, so it's not strictly necessary
    # but good practice if instances were reused or modified outside the loop).
    if model_name == "Bernoulli Naïve Bayes":
        nb_instance.setFeaturesCol("features") # Reset to default for original definition


Training Multinomial Naïve Bayes...

Training Bernoulli Naïve Bayes...

Training Complement Naïve Bayes...


In [ ]:
# ==========================================
# STEP 5: Performance Evaluation & Analysis Output
# ==========================================
print("\n" + "=" * 65)
print("          NAÏVE BAYES MODEL PERFORMANCE COMPARISON           ")
print("=" * 65)
print(f"{'Model Variant':<25} | {'Accuracy':<9} | {'Precision':<9} | {'Recall':<9} | {'F1-Score':<9}")
print("-" * 65)

best_model = None
best_f1 = 0.0

for model_name, metrics in results.items():
    print(f"{model_name:<25} | {metrics['Accuracy']:.4f}    | {metrics['Precision']:.4f}    | {metrics['Recall']:.4f}    | {metrics['F1-Score']:.4f}")
    if metrics["F1-Score"] > best_f1:
        best_f1 = metrics["F1-Score"]
        best_model = model_name

print("=" * 65)
print(f"Top Performing Model for Spam Detection: {best_model}")


          NAÏVE BAYES MODEL PERFORMANCE COMPARISON           
Model Variant             | Accuracy  | Precision | Recall    | F1-Score 
-----------------------------------------------------------------
Multinomial Naïve Bayes   | 0.9908    | 0.9817    | 0.9908    | 0.9862
Bernoulli Naïve Bayes     | 0.9908    | 0.9817    | 0.9908    | 0.9862
Complement Naïve Bayes    | 0.9908    | 0.9817    | 0.9908    | 0.9862
Top Performing Model for Spam Detection: Multinomial Naïve Bayes
